# 04 — Evaluation: Full Model Comparison

This notebook provides a comprehensive comparison of all recommender models:
- Popularity baseline
- Item-CF
- SVD
- SVD++
- Hybrid (α=0.8)

Metrics: Precision@K, Recall@K, HR@K, NDCG@K, MAP@K, RMSE, MAE

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DIR, MODELS_DIR, N_FACTORS, N_EPOCHS, ALPHA
from src.models.popularity import PopularityRecommender
from src.models.item_cf import ItemCFRecommender
from src.models.matrix_factorization import SVDRecommender
from src.models.hybrid import HybridRecommender
from src.data.preprocess import preprocess_movies
from src.evaluation.benchmark import build_ground_truth, build_user_seen_items
from src.evaluation.metrics import (
    compute_ranking_metrics, compute_rating_metrics,
    rmse as compute_rmse,
)
from src.logging_utils import eval_logger

sns.set_style('whitegrid')
eval_logger.start_phase('eval_nb', 'Evaluation notebook started')

## 1. Load Data

In [ ]:
train  = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val    = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
test   = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
movies = pd.read_parquet(PROCESSED_DIR / 'movies.parquet')
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

## 2. Train All Models

In [ ]:
print('Fitting Popularity...')
pop = PopularityRecommender()
pop.fit(train)

print('Fitting Item-CF...')
cf = ItemCFRecommender(k=50)
cf.fit(train)

print('Fitting SVD...')
svd = SVDRecommender(n_factors=N_FACTORS, n_epochs=N_EPOCHS, use_svdpp=False)
svd.fit(train)

print('Fitting SVD++...')
svdpp = SVDRecommender(n_factors=N_FACTORS, n_epochs=N_EPOCHS, use_svdpp=True)
svdpp.fit(train)

print('Fitting Hybrid...')
hybrid = HybridRecommender(cf_model=svd, fallback_model=pop, alpha=ALPHA)
hybrid.fit(train, movies)

print('All models trained.')

## 3. Evaluate All Models at K=10

In [ ]:
ground_truth = build_ground_truth(test)
seen_items   = build_user_seen_items(train)

# Use a sample of 500 users for speed
sample_users = list(ground_truth.keys())[:500]

def get_recs(model, users, n=10):
    return {
        uid: [r['movie_id'] for r in model.recommend(uid, n, seen_items.get(uid, set()))]
        for uid in users
    }

all_recs = {
    'Popularity': get_recs(pop,    sample_users),
    'Item-CF':    get_recs(cf,     sample_users),
    'SVD':        get_recs(svd,    sample_users),
    'SVD++':      get_recs(svdpp,  sample_users),
    'Hybrid':     get_recs(hybrid, sample_users),
}

rows = []
for name, recs in all_recs.items():
    m = compute_ranking_metrics(recs, ground_truth, k=10)
    rows.append({'Model': name, **m})

results_df = pd.DataFrame(rows).set_index('Model')
results_df.columns = ['P@10', 'R@10', 'HR@10', 'NDCG@10', 'MAP@10']
print(results_df.round(4).to_string())

## 4. Visualise Comparison

In [ ]:
results_df[['P@10', 'HR@10', 'NDCG@10']].plot(
    kind='bar', figsize=(12, 5),
    title='Model Comparison at K=10'
)
plt.ylabel('Score')
plt.xticks(rotation=0)

# Draw success threshold lines
from src.config import SUCCESS_HR_AT_10, SUCCESS_PRECISION_AT_10
plt.axhline(SUCCESS_HR_AT_10, color='green', linestyle='--', linewidth=1, label=f'HR@10 threshold ({SUCCESS_HR_AT_10})')
plt.axhline(SUCCESS_PRECISION_AT_10, color='orange', linestyle='--', linewidth=1, label=f'P@10 threshold ({SUCCESS_PRECISION_AT_10})')
plt.legend(bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

## 5. Rating Metrics (RMSE / MAE)

In [ ]:
rating_rows = []
for model_name, model_obj in [('SVD', svd), ('SVD++', svdpp), ('Hybrid', hybrid)]:
    if not hasattr(model_obj, 'predict_rating'):
        continue
    sample = val.head(1000)
    actuals = sample['rating'].tolist()
    preds   = [model_obj.predict_rating(int(r.user_id), int(r.movie_id))
               for r in sample.itertuples()]
    m = compute_rating_metrics(actuals, preds)
    rating_rows.append({'Model': model_name, 'RMSE': m['rmse'], 'MAE': m['mae']})

rating_df = pd.DataFrame(rating_rows).set_index('Model')
print(rating_df.round(4).to_string())

rating_df.plot(kind='bar', figsize=(8, 4), title='Rating Metrics (val sample)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

eval_logger.end_phase('eval_nb', 'Evaluation notebook complete')